# 🕹️ Deep RL Evolution Pipeline — REINFORCE → A2C → PPO

**Environment:** `ALE/Pong-v5` from raw pixels  
**Runtime:** GPU (T4 recommended) | Runtime → Change runtime type → GPU

---

## 1 · Environment Setup

Run **once** per Colab session (after a Runtime → Restart, re-run this cell first).

In [ ]:
# ============================================================
# COMPLETE ATARI + PYTORCH SETUP FOR GOOGLE COLAB
# ============================================================

# 1. System packages
!apt-get update -qq
!apt-get install -y -qq xvfb ffmpeg

# 2. Python packages
#    - Do NOT pin old Gymnasium/ALE versions (breaks ROM registration)
#    - Do NOT upgrade setuptools beyond Colab's compatible version
#    - Use autorom[accept-rom-license] for proper ROM installation
!pip install -q \
    "setuptools<82" \
    jedi \
    "gymnasium[atari]" \
    "autorom[accept-rom-license]" \
    opencv-python-headless \
    pyvirtualdisplay \
    matplotlib \
    pandas \
    imageio \
    imageio-ffmpeg

# 3. Install Atari ROMs
!AutoROM --accept-license

# 4. Verify everything
import sys
import gymnasium as gym
import ale_py
import torch
import numpy as np
import cv2
import matplotlib
import imageio
import setuptools

print('=' * 65)
print('ENVIRONMENT')
print('=' * 65)
print('Python      :', sys.version.split()[0])
print('Gymnasium   :', gym.__version__)
print('ALE         :', ale_py.__version__)
print('PyTorch     :', torch.__version__)
print('NumPy       :', np.__version__)
print('OpenCV      :', cv2.__version__)
print('Matplotlib  :', matplotlib.__version__)
print('ImageIO     :', imageio.__version__)
print('Setuptools  :', setuptools.__version__)

print('\n' + '=' * 65)
print('PYTORCH GPU')
print('=' * 65)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

print('\n' + '=' * 65)
print('ATARI TEST')
print('=' * 65)
# Register ALE namespace (required before any gym.make('ALE/...'))
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5')
obs, info = env.reset()
print('Environment :', 'ALE/Pong-v5')
print('Observation :', obs.shape)
print('Action space:', env.action_space)
env.close()

print('\n' + '=' * 65)
print('✅ SETUP COMPLETE — ROMs installed, ALE works, PyTorch ready')
print('=' * 65)

In [ ]:
import os, sys

# ── Clone / pull the project repo ─────────────────────────────────────────
REPO_URL  = 'https://github.com/YOUR_USERNAME/rl-atari-evolution.git'  # ← update
REPO_NAME = 'rl-atari-evolution'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    !git -C {REPO_NAME} pull

# Put the project root on the Python path
ROOT = os.path.abspath(REPO_NAME)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

print(f'📂 Working directory: {os.getcwd()}')

## 2 · Virtual Display
Colab has no physical display — we start Xvfb so ALE can render frames for video recording.

In [ ]:
from visualization.display import init_virtual_display

_display = init_virtual_display()
print('Virtual display ready ✓')

## 3 · Sanity Check — Wrapped Environment
Verify the full wrapper chain: shape, dtype, value range, and action space.

In [ ]:
import numpy as np
from config import CFG
from src.wrappers import make_atari_env   # ale_py is registered inside this import

env = make_atari_env(CFG.ENV_NAME, seed=CFG.SEED)
obs, info = env.reset(seed=CFG.SEED)
obs_arr = np.asarray(obs)

print(f'Environment : {CFG.ENV_NAME}')
print(f'Obs shape   : {obs_arr.shape}   (expect (4, 84, 84))')
print(f'Obs dtype   : {obs_arr.dtype}  (expect float32)')
print(f'Obs range   : [{obs_arr.min():.3f}, {obs_arr.max():.3f}]  (expect [0.0, 1.0])')
print(f'Action space: {env.action_space}  (6 discrete actions)')

# One step to verify the 5-tuple API
obs2, reward, terminated, truncated, info2 = env.step(env.action_space.sample())
print(f'Step API    : obs={np.asarray(obs2).shape}, reward={reward}, '
      f'terminated={terminated}, truncated={truncated}  ✓')
env.close()
print('\n✅ Wrapper chain OK')

## 4 · REINFORCE Training

Adjust `--episodes` to taste:
- **Quick smoke-test:** `--episodes 50` (~2 min on T4)
- **Meaningful learning:** `--episodes 3000` (2–4 h on T4 — REINFORCE is slow to converge)

Progress is printed every 10 episodes and logged to `logs/reinforce_rewards.csv`.

In [ ]:
!python -m src.train \
    --algo reinforce \
    --episodes 1000 \
    --save-freq 50 \
    --log-interval 10 \
    --seed 42

## 5 · Live Reward Plot
Plot the moving average from the CSV log to track learning progress.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = 'logs/reinforce_rewards.csv'
df = pd.read_csv(log_path)

WINDOW = 50
df['moving_avg'] = df['reward'].rolling(WINDOW, min_periods=1).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df['episode'], df['reward'],     alpha=0.3, color='steelblue', label='Episode reward')
axes[0].plot(df['episode'], df['moving_avg'], color='navy', linewidth=2, label=f'{WINDOW}-ep moving avg')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].set_title('REINFORCE — Training Rewards')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df['episode'], df['loss'], alpha=0.5, color='tomato', label='Policy loss')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Loss')
axes[1].set_title('REINFORCE — Policy Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('logs/reinforce_training_curve.png', dpi=120)
plt.show()
print('Saved → logs/reinforce_training_curve.png')

## 6 · Visual Evaluation — Gameplay Video
Load the latest checkpoint, run the agent for 20 seconds, record an `.mp4`, and display it inline.

In [ ]:
import glob, os

# Prefer numbered episode checkpoints; fall back to final.pt
checkpoints = sorted(
    glob.glob('checkpoints/reinforce/ep_*.pt'),
    key=lambda p: int(os.path.basename(p).replace('ep_', '').replace('.pt', ''))
)
if not checkpoints:
    checkpoints = glob.glob('checkpoints/reinforce/final.pt')

if not checkpoints:
    print('⚠️  No checkpoint found — run the training cell first.')
else:
    latest_ckpt = checkpoints[-1]
    print(f'Latest checkpoint: {latest_ckpt}')

In [ ]:
import os, glob
import torch
import numpy as np
import gymnasium as gym
from IPython.display import Video, display as ipy_display

from config import CFG
from src.wrappers import make_eval_env   # also registers ale_py
from src.reinforce import REINFORCEAgent

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
VIDEO_DIR = 'videos/reinforce'
MAX_STEPS = 20 * 30   # 20 seconds at ~30 fps

os.makedirs(VIDEO_DIR, exist_ok=True)

# ── Load agent ─────────────────────────────────────────────────────────────
eval_env   = make_eval_env(CFG.ENV_NAME, seed=0, render_mode='rgb_array')
action_dim = eval_env.action_space.n

agent = REINFORCEAgent(action_dim=action_dim, device=DEVICE)
agent.load(latest_ckpt)
agent.policy.eval()

# ── Wrap with RecordVideo ───────────────────────────────────────────────────
rec_env = gym.wrappers.RecordVideo(
    eval_env,
    video_folder=VIDEO_DIR,
    episode_trigger=lambda ep: ep == 0,
    name_prefix='reinforce_eval',
    video_length=MAX_STEPS,
)

obs, _    = rec_env.reset(seed=0)
done      = False
total_rew = 0.0
step      = 0

while not done and step < MAX_STEPS:
    obs_t = torch.tensor(
        np.asarray(obs, dtype=np.float32), dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        action, _ = agent.policy.get_action(obs_t)

    obs, reward, terminated, truncated, _ = rec_env.step(action)
    total_rew += float(reward)
    done = terminated or truncated
    step += 1

rec_env.close()
print(f'Episode done — steps: {step}  total reward: {total_rew:.1f}')

# ── Display inline ─────────────────────────────────────────────────────────
videos = sorted(glob.glob(f'{VIDEO_DIR}/*.mp4'))
if videos:
    print(f'Displaying: {videos[-1]}')
    ipy_display(Video(videos[-1], embed=True, width=420))
else:
    print('⚠️  No .mp4 found — ensure ffmpeg is installed and the episode ran.')

## 7 · Next Steps

| Step | Command |
|---|---|
| Train A2C | `!python -m src.train --algo a2c --episodes 3000` |
| Train PPO | `!python -m src.train --algo ppo --episodes 5000` |
| Compare all three | Run `benchmarks/evaluator.py` |

---
*Tip: REINFORCE needs many more episodes than A2C/PPO to learn Pong.  
Scores above -10 after 2 000 episodes indicate the agent is learning.*